In [1]:
import os

# Твой блок путей
USER_HOME = "/home/nshevtsova"
os.environ['HF_HOME'] = f"{USER_HOME}/.cache/huggingface"
os.environ['XDG_CACHE_HOME'] = f"{USER_HOME}/.cache"
os.environ['MPLCONFIGDIR'] = f"{USER_HOME}/.config/matplotlib"


# Создаем папки, чтобы не было Permission Denied
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)


In [2]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

from diffusers import StableDiffusionPipeline
from transformers import CLIPTextModelWithProjection, T5EncoderModel, CLIPTokenizer, T5TokenizerFast
import torch.nn.functional as F
from torchvision import models, transforms

from torchmetrics.image.fid import FrechetInceptionDistance
from prdc import compute_prdc
import open_clip
import random

from cleanfid import fid as clean_fid
import tempfile
import shutil
import os


/opt/conda/envs/lora/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/envs/lora/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
#METADATA = Path("/home/nshevtsova/datasets/BCN_org/metadata.csv")
#IMG_DIR = Path("/home/nshevtsova/datasets/BCN_org")

METADATA = Path("/home/nshevtsova/metadata_clean.csv")
IMG_DIR = Path("/home/nshevtsova/BCN_clean")

DATASET_LORA_DIR = "/home/nshevtsova/datasets/bio_clip"

run_name = "adamw_run" # МЕНЯЕШЬ ЗДЕСЬ
LOG_FILE_PATH = f"{run_name}_metrics.txt"

MIN_REAL = 100
N_GEN = 100

df = pd.read_csv(METADATA)


In [3]:
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None
).to("cuda")

pipe.load_lora_weights(
    f"lora_output/{run_name}", 
    weight_name="last.safetensors"
)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()


Loading pipeline components...: 100%|██████████| 6/6 [00:10<00:00,  1.67s/it]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


In [5]:
import random
from glob import glob

# Словарь для хранения загруженных промптов по классам
PROMPTS_CACHE = {}

def load_prompts_for_class(cls):
    """
    Загружает промпты, обеспечивая одинаковый порядок файлов.
    """
    if cls in PROMPTS_CACHE:
        return PROMPTS_CACHE[cls]
    
    class_dir = os.path.join(DATASET_LORA_DIR, cls)
    if not os.path.exists(class_dir):
        return []

    # КРИТИЧНО: sorted() гарантирует, что список промптов всегда будет в одном порядке
    txt_files = sorted(glob(os.path.join(class_dir, "*.txt")))
    
    prompts = []
    for txt_file in txt_files:
        try:
            with open(txt_file, "r", encoding="utf-8") as f:
                prompt = f.read().strip()
                if prompt:
                    prompts.append(prompt)
        except Exception as e:
            print(f"Ошибка чтения файла {txt_file}: {e}")
            
    PROMPTS_CACHE[cls] = prompts
    return prompts

def generate_images(pipe, cls, n, seed=1908):
    """
    Генерируем картинки: промпты выбираются строго по seed, 
    но сами изображения остаются 'случайными'.
    """
    prompts_pool = load_prompts_for_class(cls)
    
    # Создаем локальный рандом, привязанный к seed. 
    # Он будет отвечать ТОЛЬКО за выбор промптов в этой функции.
    prompt_random = random.Random(seed)
    
    negative = "clinical photo, patient skin, ruler, green, text, hair, watermark, low quality, blurry"
    #negative_prompt = "clinical photo, patient skin, ruler, green, text, hair, cosmos, blue, abstract, oversaturated, vibrant colors, electric colors, neon red, painting, illustration, cartoon, number, watermark, low quality, blurry"
 
    images = []
    used_prompts = []

    for i in range(n):
        # Теперь .choice() всегда будет выбирать один и тот же промпт при одинаковом seed
        if prompts_pool:
            current_prompt = prompt_random.choice(prompts_pool)
        else:
            current_prompt = f"dx_{cls.lower()}, dermoscopy image"

        with torch.no_grad(), torch.amp.autocast('cuda'):
            img = pipe(
                current_prompt,
                negative_prompt=negative,
                num_images_per_prompt=1,
                guidance_scale=8,        
                num_inference_steps=30,    
                cross_attention_kwargs={"scale": 0.7}
            ).images[0]

        images.append(img)
        used_prompts.append(current_prompt)

    return images, used_prompts
    

In [6]:
def load_real_images(df, image_dir, cls, n=5):
    df = df.sort_values("isic_id") 
    df_cls = df[df["class"] == cls]

    if len(df_cls) == 0:
        return [], []

    # Фиксируем random_state для воспроизводимости
    df_cls = df_cls.sample(n=min(n, len(df_cls)), random_state=1908)

    images = []
    ids = []

    for isic_id in df_cls["isic_id"]:
        img_path = Path(image_dir) / f"{isic_id}.jpg"

        if img_path.exists():
            images.append(Image.open(img_path).convert("RGB"))
            ids.append(isic_id) # Сохраняем ID, чтобы найти потом .txt

    print(f"Loaded images → real: {len(images)} for class {cls}")
    return images, ids

In [7]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

fid = FrechetInceptionDistance(feature=2048).to(device)

def pil_to_uint8_tensor(img):
    # PIL → uint8 tensor [3,H,W]
    x = np.array(img, dtype=np.uint8)
    x = torch.from_numpy(x).permute(2, 0, 1)  # HWC → CHW
    return x

def compute_fid(real_imgs, fake_imgs):
    fid_metric = FrechetInceptionDistance(feature=2048).to(device)

    for img in real_imgs:
        x = pil_to_uint8_tensor(img).unsqueeze(0).to(device)
        fid_metric.update(x, real=True)

    for img in fake_imgs:
        x = pil_to_uint8_tensor(img).unsqueeze(0).to(device)
        fid_metric.update(x, real=False)

    fid_value = fid_metric.compute().item()
    return fid_value

In [8]:
# для FID
inception_feat = models.inception_v3(
    weights=models.Inception_V3_Weights.IMAGENET1K_V1,
    transform_input=False
).to(device).eval()
inception_feat.fc = torch.nn.Identity()


In [9]:
def extract_features(images):
    feats = []
    preprocess = transforms.Compose([
        # Сначала уменьшаем короткую сторону до 299
        transforms.Resize(299), 
        # Вырезаем центральный квадрат 299x299 (без искажений пропорций)
        transforms.CenterCrop(299), 
        transforms.ToTensor(),
        # InceptionV3 ожидает нормализацию ImageNet
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    with torch.no_grad():
        for img in images:
            x = preprocess(img).unsqueeze(0).to(device)
            f = inception_feat(x)[0].detach().cpu().numpy()
            feats.append(f)

    return np.vstack(feats)


In [10]:
def compute_prd(real_imgs, fake_imgs):
    real_feats = extract_features(real_imgs)
    fake_feats = extract_features(fake_imgs)

    return compute_prdc(
        real_features=real_feats,
        fake_features=fake_feats,
        nearest_k=5
    )


In [11]:
os.environ['HF_TOKEN'] = "" 

In [12]:
from open_clip import create_model_from_pretrained, get_tokenizer

MY_CACHE_DIR = os.path.join(USER_HOME, ".cache", "clip")
os.makedirs(MY_CACHE_DIR, exist_ok=True)

model_id = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'

print("Loading BiomedCLIP...")
model, preprocess = create_model_from_pretrained(model_id)
tokenizer = get_tokenizer(model_id)

model.to(device)
model.eval()

context_length = 256 # Специфично для BiomedCLIP

Loading BiomedCLIP...


/opt/conda/envs/lora/lib/python3.10/site-packages/open_clip/factory.py:128: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_locati

In [13]:
def clip_similarity(images, texts):
    # Используем глобальные объекты из твоей ячейки инициализации
    global model, tokenizer, preprocess, device, context_length
    
    # 1. Токенизация списка текстов
    # Проверяем, что texts — это список строк
    text_tokens = tokenizer(texts, context_length=context_length).to(device)
    
    # 2. Подготовка картинок батчем (используем твое имя 'preprocess')
    # Это превратит список PIL-картинок в один тензор [100, 3, 224, 224]
    image_input = torch.stack([preprocess(img) for img in images]).to(device)
    
    with torch.no_grad(), torch.amp.autocast('cuda'):
        # Получаем эмбеддинги
        image_features = model.encode_image(image_input)
        text_features = model.encode_text(text_tokens)
        
        # Нормализация для расчета Cosine Similarity
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        
        # Считаем сходство пар (image[i] vs text[i])
        # .sum(dim=-1) по нормализованным векторам — это и есть косинусное сходство
        similarities = (image_features * text_features).sum(dim=-1)
        
    return similarities.mean().item()

In [14]:
# Вспомогательная функция для записи в файл и вывода в консоль
def log_and_print(message):
    print(message)
    with open(LOG_FILE_PATH, "a", encoding="utf-8") as f:
        f.write(message + "\n")

# Инициализируем файл (очищаем старый, если был)
with open(LOG_FILE_PATH, "w", encoding="utf-8") as f:
    f.write(f"=== METRICS FOR RUN: {run_name} ===\n")

results = []

DIAGNOSIS_MAP = {
    "Seborrheic keratosis": "BKL",
    "Solar lentigo": "BKL",
    "Melanoma metastasis": "MEL",
    "Melanoma, NOS": "MEL",
    "Dermatofibroma": "DF",
    "Solar or actinic keratosis": "AK",
    "Basal cell carcinoma": "BCC",
    "Nevus": "NV",
    "Squamous cell carcinoma, NOS": "SCC",
}

df = pd.read_csv(METADATA)
df["class"] = df["diagnosis_3"].map(DIAGNOSIS_MAP)
df = df.dropna(subset=["class"])
classes = sorted(df["class"].unique())

log_and_print(f"Found {len(classes)} classes: {classes}")

for cls in classes:
    log_and_print(f"\n" + "="*30)
    log_and_print(f"=== Class {cls} ===")
    log_and_print("="*30)

    real_imgs, real_ids = load_real_images(
        df=df,
        image_dir=IMG_DIR,
        cls=cls,
        n=N_GEN
    )

    if len(real_imgs) < 2:
        log_and_print(f"[SKIP] Not enough real images for {cls}")
        continue

    real_prompts = []
    trigger = f"dx_{cls.lower()}, "
    for isic_id in real_ids:
        txt_path = os.path.join(DATASET_LORA_DIR, cls, f"{isic_id}.txt")
        try:
            with open(txt_path, "r", encoding="utf-8") as f:
                p = f.read().strip()
                real_prompts.append(p.replace(trigger, "").strip())
        except FileNotFoundError:
            log_and_print(f"Внимание: Текст для {isic_id} не найден по пути {txt_path}")
            real_prompts.append(f"dermoscopy image of {cls}")

    # Генерация (использует нашу функцию с фиксированным выбором промптов)
    fake_imgs, fake_prompts = generate_images(pipe, cls, n=N_GEN, seed=42)

    clean_prompts = [p.replace(trigger, "") for p in fake_prompts]

    # Расчет метрик
    fid = compute_fid(real_imgs, fake_imgs)
    prd = compute_prd(real_imgs, fake_imgs)
    clip_sim = clip_similarity(fake_imgs, clean_prompts)
    clip_sim_real = clip_similarity(real_imgs, real_prompts)

    results.append({
        "class": cls,
        "n_real": len(real_imgs),
        "fid": fid,
        "precision": prd["precision"],
        "recall": prd["recall"],
        "density": prd["density"],     
        "coverage": prd["coverage"],
        "clip": clip_sim,
        "clip_real": clip_sim_real,
    })

    # Формируем красивый отчет для записи
    metrics_report = (
        f"FID: {fid:.2f}\n"
        f"Precision: {prd['precision']:.3f}\n"
        f"Recall: {prd['recall']:.3f}\n"
        f"Density: {prd['density']:.3f}\n"
        f"Coverage: {prd['coverage']:.3f}\n"
        f"CLIP (Fake): {clip_sim:.3f}\n"
        f"CLIP (Real): {clip_sim_real:.3f}\n"
    )
    log_and_print(metrics_report)

    del fake_imgs
    torch.cuda.empty_cache()

log_and_print("\n=== ALL CLASSES FINISHED ===")

Found 7 classes: ['AK', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCC']

=== Class AK ===
Loaded images → real: 100 for class AK


100%|██████████| 30/30 [00:07<00:00,  4.22it/s]


Num real: 100 Num fake: 100
FID: 158.16
Precision: 0.580
Recall: 0.420
Density: 0.434
Coverage: 0.430
CLIP: 0.433
CLIP (Real): 0.429

=== Class BCC ===
Loaded images → real: 100 for class BCC


100%|██████████| 30/30 [00:07<00:00,  4.10it/s]


Num real: 100 Num fake: 100
FID: 172.52
Precision: 0.750
Recall: 0.310
Density: 0.506
Coverage: 0.470
CLIP: 0.450
CLIP (Real): 0.449

=== Class BKL ===
Loaded images → real: 100 for class BKL


...

Num real: 100 Num fake: 100
FID: 216.20
Precision: 0.910
Recall: 0.310
Density: 0.836
Coverage: 0.430
CLIP: 0.432
CLIP (Real): 0.447

=== Class NV ===
Loaded images → real: 100 for class NV


100%|██████████| 30/30 [00:10<00:00,  2.73it/s]


Num real: 100 Num fake: 100
FID: 187.82
Precision: 0.720
Recall: 0.550
Density: 0.436
Coverage: 0.490
CLIP: 0.437
CLIP (Real): 0.459

=== Class SCC ===
Loaded images → real: 100 for class SCC


100%|██████████| 30/30 [00:10<00:00,  2.95it/s]


Num real: 100 Num fake: 100
FID: 177.97
Precision: 0.580
Recall: 0.280
Density: 0.484
Coverage: 0.410
CLIP: 0.441
CLIP (Real): 0.436


In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from torchvision import models, transforms
from torchmetrics.image.fid import FrechetInceptionDistance
from prdc import compute_prdc
import gc

# --- КОНФИГУРАЦИЯ ---
USER_HOME = "/home/nshevtsova"
REAL_IMG_DIR = Path("/home/nshevtsova/BCN_clean")
REAL_METADATA_PATH = Path("/home/nshevtsova/metadata_clean.csv")
SYNTH_BASE_DIR = Path("/home/nshevtsova/synthetic_dataset")
SYNTH_RUNS = ["adamw_run", "gamma_run", "78_run"] 

SEED = 1908
LOG_DIR = Path("/home/nshevtsova/metrics_eval")
os.makedirs(LOG_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DIAGNOSIS_MAP = {
    "Solar or actinic keratosis": "AK", "Basal cell carcinoma": "BCC",
    "Seborrheic keratosis": "BKL", "Solar lentigo": "BKL",
    "Dermatofibroma": "DF", "Melanoma metastasis": "MEL",
    "Melanoma, NOS": "MEL", "Nevus": "NV",
    "Squamous cell carcinoma, NOS": "SCC",
}

# Кэш для эмбеддингов реальных изображений (чтобы не пересчитывать 4k фото каждый раз)
REAL_FEATURES_CACHE = {}

# --- ПОДГОТОВКА МОДЕЛИ ---
print("Loading InceptionV3...")
inception_model = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)
inception_model.fc = torch.nn.Identity()
inception_model.to(device).eval()

preprocess = transforms.Compose([
    transforms.Resize(299), transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ---

def get_features(image_paths):
    """Извлекает признаки для списка путей"""
    feats = []
    with torch.no_grad():
        for p in tqdm(image_paths, desc="Extracting features", leave=False):
            img = Image.open(p).convert("RGB")
            tensor = preprocess(img).unsqueeze(0).to(device)
            f = inception_model(tensor).cpu().numpy()
            feats.append(f)
    return np.vstack(feats)

def pil_to_uint8_tensor(path):
    img = Image.open(path).convert("RGB")
    return torch.from_numpy(np.array(img, dtype=np.uint8)).permute(2, 0, 1)

# --- ОСНОВНОЙ ЦИКЛ ---

df_real_all = pd.read_csv(REAL_METADATA_PATH)
df_real_all["class"] = df_real_all["diagnosis_3"].map(DIAGNOSIS_MAP)

for run in SYNTH_RUNS:
    run_path = SYNTH_BASE_DIR / run
    metadata_path = run_path / "metadata_synth.csv"
    
    if not metadata_path.exists():
        continue
    
    print(f"\n>>> Evaluating run: {run}")
    df_synth = pd.read_csv(metadata_path)
    counts = df_synth["diagnosis_3"].value_counts()
    valid_classes = counts[counts > 100].index.tolist()
    
    run_results = []
    
    for cls in valid_classes:
        print(f"  Class: {cls} (Synth: {counts[cls]})")
        
        # 1. Пути синтетики
        s_ids = df_synth[df_synth["diagnosis_3"] == cls]["isic_id"].tolist()
        synth_paths = [run_path / f"{sid}.jpg" for sid in s_ids]
        
        # 2. Реальные данные: ПОЛНЫЙ ПУЛ (для PRDC)
        df_cls_real_all = df_real_all[df_real_all["class"] == cls].sort_values("isic_id")
        real_paths_all = [REAL_IMG_DIR / f"{rid}.jpg" for rid in df_cls_real_all["isic_id"]]
        real_paths_all = [p for p in real_paths_all if p.exists()]

        # 3. Реальные данные: SAMPLE (для FID)
        n_fid = min(len(real_paths_all), len(synth_paths))
        df_cls_real_sampled = df_cls_real_all.sample(n=n_fid, random_state=SEED)
        real_paths_fid = [REAL_IMG_DIR / f"{rid}.jpg" for rid in df_cls_real_sampled["isic_id"]]

        # --- РАСЧЕТ FID ---
        fid_metric = FrechetInceptionDistance(feature=2048).to(device)
        for p in real_paths_fid:
            fid_metric.update(pil_to_uint8_tensor(p).unsqueeze(0).to(device), real=True)
        for p in synth_paths:
            fid_metric.update(pil_to_uint8_tensor(p).unsqueeze(0).to(device), real=False)
        fid_v = fid_metric.compute().item()
        
        # --- РАСЧЕТ PRDC ---
        # Проверяем кэш для реальных признаков (все 4000 штук)
        if cls not in REAL_FEATURES_CACHE:
            print(f"    Encoding {len(real_paths_all)} real images for cache...")
            REAL_FEATURES_CACHE[cls] = get_features(real_paths_all)
        
        real_feats = REAL_FEATURES_CACHE[cls]
        synth_feats = get_features(synth_paths)
        
        prdc_v = compute_prdc(real_features=real_feats, fake_features=synth_feats, nearest_k=5)
        
        # --- СБОР РЕЗУЛЬТАТОВ ---
        run_results.append({"class": cls, "fid": fid_v, **prdc_v})
        print(f"    FID: {fid_v:.2f} | P: {prdc_v['precision']:.3f} | R: {prdc_v['recall']:.3f} | D: {prdc_v['density']:.3f} | C: {prdc_v['coverage']:.3f}")
        
        del fid_metric, synth_feats
        torch.cuda.empty_cache()
        gc.collect()

    if run_results:
        pd.DataFrame(run_results).to_csv(LOG_DIR / f"metrics_{run}.csv", index=False)

/opt/conda/envs/lora/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading InceptionV3...

>>> Evaluating run: adamw_run
  Class: NV (Synth: 941)
    Encoding 4706 real images for cache...


Num real: 4706 Num fake: 941
    FID: 134.21 | P: 0.465 | R: 0.556 | D: 0.227 | C: 0.084
  Class: MEL (Synth: 776)
    Encoding 3882 real images for cache...


Num real: 3882 Num fake: 776
    FID: 149.87 | P: 0.617 | R: 0.302 | D: 0.386 | C: 0.099
  Class: BCC (Synth: 575)
